# Do `canna.lisa`'s noise and galactic binary share a convention?

This notebook builds the **datastream and the noise with `canna.lisa`'s own functions**
(`LisaGB.clean_signal`, `LisaGB.sample_observation`, `LisaGB.noise_psd`,
`LisaGB.log_likelihood`, `LisaGB.preprocess`), on the grid and with the injection of
**arXiv:2606.20269 Sec. IV.1**, and then reproduces its SNR $=20.7$ and its Fig. 6 corner.

The paper's own study code is public — `docs/studies/lisa/` in
[pywavelet/wdm_transform](https://github.com/pywavelet/wdm_transform) — so every claim
below is checked against it rather than guessed.

**What comes out (all verified in the cells below):**

1. **Same instrument, same waveform.** `LisaGB.noise_psd` is the paper's
   `noise_tdi15_psd` — the same two-parameter (3 fm/s², 15 pm) TDI-1.5 A/E/T model — and
   the GB is the same `jaxgb`, TDI 1.5, AET, equal-arm orbits.
2. **`noise_psd` is now in fractional frequency**, the units `jaxgb.get_tdi` returns.
   That is a change: it used to be a *fractional-length* PSD, out by
   $4x^2$, $x=2\pi fL/c$, against the response it whitens. §1 checks the new expression
   against the textbook TDI-1.5 formula to machine precision, §5 checks the pairing with
   the response itself.
3. **Inside `canna.lisa` the datastream and the noise share a convention.**
   `sample_observation`, `log_likelihood` and `preprocess` all use
   $\mathbb{E}|n_k|^2 = S_k T/2$; the whitened residual has $\chi^2/\mathrm{dof}=1$.
4. **`LisaGB.snr` is still not the matched-filter SNR** — it is
   $\sqrt{\langle h|h\rangle/2}$, i.e. low by $\sqrt2$. Left as it was; it is a constant
   relabelling, unlike the units, and changing it is a separate call.
5. **The paper's 20.7 is the physical SNR times two convention factors.** The injection
   quoted there, $A=5.35\times10^{-24}$, really has $\sqrt{\langle h|h\rangle}=0.85$.
   The paper's `_optimal_snr_sq` weights with `4*df*dt**2` — right for a bare
   `np.fft.rfft`, but applied to a `jaxgb` template that is already a physical
   $\tilde h$ — and it whitens that template with a fractional-length PSD. So
   $20.7 = \Delta t \cdot 2x \cdot 0.85$ with $\Delta t = 166.7$ s and $2x = 0.145$.
   Both factors sit in its injected data as well, so its WDM-vs-Fourier comparison is
   unaffected — its effective injection is just louder than the quoted $A$.

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import jax

jax.config.update("jax_enable_x64", True)  # the LISA band needs float64 end to end

import corner
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np

from canna.lisa import LisaGB
from canna.lisa.constants import SPEED_OF_LIGHT
from canna.lisa.priors import chirp_mass_from_fdot

# ── arXiv:2606.20269, Sec. IV.1 ──────────────────────────────────────────────
# grid: dt = 1/(2 f_max) with f_max = 3 mHz, N = 2**17 samples -> T ~ 253 d
DT = 1.0 / (2.0 * 3.0e-3)
N_SAMPLES = 131072
T_OBS = N_SAMPLES * DT

# the representative seed of Fig. 6
F0, FDOT, AMP, PHI0 = 1.38599e-3, 9.39e-15, 5.35e-24, 1.74
LON, LAT, PSI, IOTA = 4.50, 0.98, 2.70, 0.58  # ecliptic lambda/beta, polarisation, inclination
SNR_PAPER = 20.7

print(f"dt      = {DT:.4f} s   (paper quotes 166.7)")
print(f"N       = {N_SAMPLES}")
print(f"T_obs   = {T_OBS:.6e} s = {T_OBS / 86400:.2f} d   (paper quotes ~253 d)")
print(f"f_nyq   = {0.5 / DT:.4e} Hz")
print(f"1/T     = {1 / T_OBS:.4e} Hz")

## 0. The problem, on the paper's grid

`LisaGB` slides a window of `wdm_freq_bands * wdm_times//2` rFFT bins over the band; the
paper instead analyses a local Fourier patch around the source. A 4096-bin window
(≈ 190 nHz wide) centred on $f_0$ holds the whole 512-bin response with room to spare, so
the two cover the same information — the source is a few bins wide and everything outside
contributes nothing to $\langle h|h\rangle$.

Two column conventions have to be pre-inverted when handing a source to `LisaGB`:

* **column 1 is a chirp mass**, which `clean_signal` maps to $\dot f$ — so invert with
  `chirp_mass_from_fdot`;
* **column 6 is a latitude**, which `clean_signal` maps to `jaxgb`'s inclination as
  $\pi/2 - \texttt{col6}$ — so pass $\pi/2 - \iota$.

`LisaGB`'s `sky_lon`/`sky_lat` go straight into `jaxgb`'s columns 3/4, exactly as the
paper's `_gb_params` does with its `ra`/`dec`, so the paper's ecliptic $(\lambda,\beta)$
go in unconverted.

In [ ]:
problem = LisaGB(
    n_sources=1,
    t_obs=T_OBS,
    sampling_step=DT,
    wdm_freq_bands=256,
    wdm_times=32,
    response_points=512,
    f0_range=(1.3e-3, 1.5e-3),
)
F_WINDOW = jnp.asarray(F0)  # slide the window onto the source
FREQS = np.asarray(problem.window_freqs(F_WINDOW))


def source(f0=F0, fdot=FDOT, amp=AMP, phi0=PHI0):
    """The paper's source as a LisaGB physical row, with cols 1 and 6 pre-inverted."""
    mc = chirp_mass_from_fdot(jnp.asarray(fdot), jnp.asarray(f0))
    return jnp.stack(
        [
            jnp.asarray(f0),
            mc,
            jnp.asarray(amp),
            jnp.asarray(LON),
            jnp.asarray(LAT),
            jnp.asarray(PSI),
            jnp.pi / 2 - jnp.asarray(IOTA),  # latitude -> jaxgb inclination
            jnp.asarray(phi0),
        ]
    )[None]


p_paper = source()
print(f"window          : bins {int(problem.window_start(F_WINDOW))}..."
      f"{int(problem.window_start(F_WINDOW)) + problem.window_bins}"
      f"  ({FREQS[0]:.6e} .. {FREQS[-1]:.6e} Hz)")
print(f"f0 inside window: {FREQS[0] < F0 < FREQS[-1]}")
print(f"response        : {problem.response_points} bins around kmin="
      f"{int(problem.response.get_kmin(jnp.array([F0]))[0])}")
print(f"implied Mc      : {float(p_paper[0, 1]):.2f} Msun")
print("   -- not a double white dwarf: fdot=9.39e-15 at 1.386 mHz is a deliberately")
print("      heavy chirp, chosen so that fdot is measurable rather than prior-dominated.")

## 1. Is it the same noise PSD?

Below is the paper's PSD copied verbatim from `docs/studies/lisa/lisa_common.py`. It is a
**fractional-length** PSD: the two single-link terms are divided by the arm $L$, and the
TDI-1.5 prefactor is $4\sin(x)\,x$ with $x = 2\pi fL/c$.

`LisaGB.noise_psd` carries the same instrument model in **fractional frequency** — the
units `jaxgb.get_tdi` returns. A link measures a doppler shift, so a displacement noise
enters as $(2\pi f/c)^2$ in power instead of $1/L^2$, and the TDI prefactor is
$16\sin^2 x$. The two differ by exactly $4x^2$ (up to the paper's $x/\sin x$), and the
result is the textbook TDI-1.5 A/E/T PSD:

$$S_{A,E} = 8\sin^2x\,[\,4(1+c+c^2)\,S_{\rm acc} + (2+c)\,S_{\rm oms}\,], \qquad
S_T = 16\sin^2x\,(1-c)\,[\,2(1-c)\,S_{\rm acc} + S_{\rm oms}\,]$$

with $c=\cos x$, $S_{\rm oms} = (2\pi f/c)^2P_{\rm oms}$ and
$S_{\rm acc} = P_{\rm acc}/[(2\pi f)^2c^2]$.

In [ ]:
C_LIGHT, L_ARM = 299792458.0, 2.5e9


def _ntilda_e(f, A=3.0, P=15.0, L=L_ARM):
    """lisa_common.py::_ntilda_e -- A/E single-link term, verbatim."""
    fstar = 1.0 / (2.0 * np.pi * L / C_LIGHT)
    return 0.5 * (2.0 + np.cos(f / fstar)) * (P / L) ** 2 * 1e-24 * (
        1.0 + (0.002 / f) ** 4
    ) + 2.0 * (1.0 + np.cos(f / fstar) + np.cos(f / fstar) ** 2) * (A / L) ** 2 * 1e-30 * (
        1.0 + (0.0004 / f) ** 2
    ) * (1.0 + (f / 0.008) ** 4) * (1.0 / (2.0 * np.pi * f)) ** 4


def _ntilda_t(f, A=3.0, P=15.0, L=L_ARM):
    """lisa_common.py::_ntilda_t -- T single-link term, verbatim."""
    fstar = 1.0 / (2.0 * np.pi * L / C_LIGHT)
    return (1.0 - np.cos(f / fstar)) * (P / L) ** 2 * 1e-24 * (
        1.0 + (0.002 / f) ** 4
    ) + 2.0 * (1.0 - np.cos(f / fstar)) ** 2 * (A / L) ** 2 * 1e-30 * (
        1.0 + (0.0004 / f) ** 2
    ) * (1.0 + (f / 0.008) ** 4) * (1.0 / (2.0 * np.pi * f)) ** 4


def paper_psd(f):
    """lisa_common.py::noise_tdi15_psd, stacked A/E/T -- fractional length."""
    f = np.asarray(f, dtype=float)
    fstar = 1.0 / (2.0 * np.pi * L_ARM / C_LIGHT)
    tdi15 = 4.0 * np.sin(f / fstar) * f / fstar
    return np.stack([_ntilda_e(f) * tdi15, _ntilda_e(f) * tdi15, _ntilda_t(f) * tdi15], -1)


def textbook_psd(f):
    """TDI-1.5 A/E/T in fractional frequency, Babak+2021 -- what noise_psd should be."""
    f = np.asarray(f, dtype=float)
    x = 2.0 * np.pi * f * L_ARM / C_LIGHT
    c = np.cos(x)
    s_oms = (2.0 * np.pi * f / C_LIGHT) ** 2 * (15e-12) ** 2 * (1 + (2e-3 / f) ** 4)
    s_acc = (3e-15) ** 2 * (1 + (0.4e-3 / f) ** 2) * (1 + (f / 8e-3) ** 4) / (
        (2.0 * np.pi * f) ** 2 * C_LIGHT**2
    )
    s_ae = 8 * np.sin(x) ** 2 * (4 * (1 + c + c**2) * s_acc + (2 + c) * s_oms)
    s_t = 16 * np.sin(x) ** 2 * (1 - c) * (2 * (1 - c) * s_acc + s_oms)
    return np.stack([s_ae, s_ae, s_t], -1)


f_scan = np.logspace(-4, np.log10(3e-3), 400)
S_canna = np.asarray(problem.noise_psd(jnp.asarray(f_scan)))
S_paper, S_book = paper_psd(f_scan), textbook_psd(f_scan)
x_scan = 2.0 * np.pi * f_scan * L_ARM / C_LIGHT

print(f"max |noise_psd / textbook - 1|            : {np.abs(S_canna / S_book - 1).max():.2e}")
print(f"max |noise_psd / (paper x 4x^2) - 1|      : "
      f"{np.abs(S_canna / (S_paper * 4 * x_scan[:, None] ** 2) - 1).max():.2e}   (= 1 - sin x/x)")
print(f"S_A at f0 = {F0 * 1e3:.5f} mHz  : canna {float(problem.noise_psd(F_WINDOW)[0]):.4e}"
      f"   paper {paper_psd(F0)[0]:.4e}   ratio 4x^2 = {4 * (2 * np.pi * F0 * L_ARM / C_LIGHT) ** 2:.4e}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for i, (nm, col) in enumerate(zip("AET", ["C0", "C1", "C2"])):
    ax[0].loglog(f_scan, S_canna[:, i], col, label=f"noise_psd {nm}")
    ax[0].loglog(f_scan, S_book[:, i], "k--", lw=0.8)
    ax[1].loglog(f_scan, S_canna[:, i] / S_paper[:, i], col, label=nm)
ax[1].loglog(f_scan, 4 * x_scan**2, "k--", lw=0.9, label=r"$4x^2$")
ax[0].set(xlabel="f [Hz]", ylabel=r"$S(f)$", title="noise_psd (colour) vs textbook (dashed)")
ax[0].legend(fontsize=8)
ax[1].set(xlabel="f [Hz]", ylabel="canna / paper", title="the units factor")
ax[1].legend(fontsize=8)
fig.tight_layout()

## 2. Datastream and noise, straight from `canna.lisa`

`sample_observation` = `clean_signal` + a complex Gaussian draw with
$\mathbb{E}|n_k|^2 = S_k\,T/2$ — the correct variance for
$\tilde h_k = \Delta t\sum_j h_j e^{-2\pi i jk/N} \simeq \int h\,e^{-2\pi ift}\,dt$.
The one-sided density of a physical $\tilde h$ is $2|\tilde h|^2/T$ (the paper's
`_one_sided_density`, which is $2\Delta t|X_k|^2/N$ for a bare rFFT $X_k$).

These panels are in `canna`'s units, so they are **not** Fig. 5b: there the source peaks
near $10^{-39}$ over a $\sim3\times10^{-41}$ floor, whereas the honest picture is a source
well under the noise — $\sqrt{\langle h|h\rangle}=0.85$ at the quoted $A$. The print block
applies the two factors of §5 and lands back on Fig. 5b's numbers.

In [ ]:
clean = problem.clean_signal(p_paper, F_WINDOW)
S_win = np.asarray(problem.noise_psd(jnp.asarray(FREQS)))
data_demo = problem.sample_observation(jr.key(20269), p_paper, F_WINDOW)


def one_sided(x):
    """Physical one-sided density of a strain density h~ (the paper's 2 dt |rfft|^2 / N)."""
    return 2.0 * np.abs(np.asarray(x)) ** 2 / T_OBS


fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
band = np.logspace(-4, np.log10(3e-3), 400)
ax[0].loglog(FREQS, one_sided(data_demo[:, 0]), color="#B8B8B8", lw=0.7, label="A data")
ax[0].loglog(band, np.asarray(problem.noise_psd(jnp.asarray(band)))[:, 0],
             color="#172919", lw=2.0, label="Instrument PSD")
ax[0].loglog(FREQS, one_sided(clean[:, 0]), color="#ff7f0e", lw=1.4, label="Injected source")
ax[0].axvspan(FREQS[0], FREQS[-1], color="0.9", zorder=0)
ax[0].set(xlabel="frequency [Hz]", ylabel=r"$S^A(f)$ [strain$^2$/Hz]",
          ylim=(1e-48, 1e-38), title="analysis band, window shaded")
ax[0].legend(fontsize=8, loc="lower left")

sel = np.abs(FREQS - F0) < 4e-6
ax[1].semilogy(FREQS[sel] * 1e3, one_sided(data_demo[sel, 0]), color="#B8B8B8", lw=0.9)
ax[1].semilogy(FREQS[sel] * 1e3, S_win[sel, 0], color="#172919", lw=2.0)
ax[1].semilogy(FREQS[sel] * 1e3, one_sided(clean[sel, 0]), color="#ff7f0e", lw=1.6)
ax[1].set(xlabel="frequency [mHz]", ylabel=r"$S^A(f)$", ylim=(1e-48, 1e-41),
          title="zoom on the source")
fig.tight_layout()

x0 = 2.0 * np.pi * F0 * L_ARM / C_LIGHT
peak = one_sided(clean[:, 0]).max()
floor = S_win[np.abs(FREQS - F0).argmin(), 0]
print(f"{'':34}{'canna':>12}{'x paper convention':>20}{'Fig. 5b':>10}")
print(f"{'instrument PSD at f0':<34}{floor:>12.3e}{floor / (4 * x0**2):>20.3e}{'~3e-41':>10}")
print(f"{'source one-sided density, peak':<34}{peak:>12.3e}{peak * DT**2:>20.3e}{'~1e-39':>10}")
print("\n  PSD  / 4x^2 : fractional frequency -> the paper's fractional length")
print("  source x dt^2: physical h~ -> the bare-rfft normalisation its weight assumes")

## 3. Do the datastream and the noise agree *inside* `canna.lisa`?

Yes. Three independent checks that the drawn noise, the likelihood and the network's
whitening all read the PSD the same way:

* the ensemble periodogram of the draw over $S\,T/2$ is 1;
* $-\log\mathcal{L}$ at the truth equals the number of complex bins × channels
  ($\chi^2/\mathrm{dof} = 1$: `log_likelihood` is $-\sum|r|^2/(S T/2)$, so each complex bin
  contributes 1);
* `preprocess` — the whitening the network actually sees — gives a unit-variance image.

In [ ]:
keys = jr.split(jr.key(0), 512)
faint = source(amp=0.0)  # noise only
draws = jax.vmap(lambda k: problem.sample_observation(k, faint, F_WINDOW))(keys)
ratio = np.asarray(jnp.mean(jnp.abs(draws) ** 2, axis=0)) / (S_win * T_OBS / 2.0)
print(f"1. <|n|^2> / (S T/2)      : {ratio.mean():.4f}  (A/E/T: {ratio.mean(0)})")

n_bins = problem.window_bins * 3
obs = problem.sample_observation(jr.key(11), p_paper, F_WINDOW)
chi2 = -float(problem.log_likelihood(p_paper, obs, F_WINDOW))
print(f"2. -logL(truth)/(bins*ch) : {chi2 / n_bins:.4f}   ({chi2:.1f} / {n_bins})")

image = problem.preprocess(problem.sample_observation(jr.key(3), faint, F_WINDOW), F_WINDOW)
# preprocess ends in arcsinh, so undo it: the WDM pixels themselves must be unit variance
print(f"3. std of sinh(preprocess()): {float(jnp.sinh(image).std()):.4f}   shape {image.shape}")
print("\n-> within canna.lisa the injection, the likelihood and the network whitening")
print("   are mutually consistent. The issues below are about the absolute scale.")

## 4. The SNR ladder up to the paper's 20.7

For a one-sided PSD $S$ and a physical $\tilde h$,

$$\rho^2 \;=\; \langle h|h\rangle \;=\; \frac{4}{T}\sum_k \frac{|\tilde h_k|^2}{S_k}
\;=\; 2\sum_k \frac{|\tilde h_k|^2}{S_k T/2},$$

while `LisaGB.snr` returns $\sqrt{\sum_k |\tilde h_k|^2/(S_kT/2)}$ — **low by $\sqrt2$**,
and not a PSD error: the same $S_kT/2$ appears in `log_likelihood`, and
$\rho^2 = 2\,[\log\mathcal{L}(h)-\log\mathcal{L}(0)]$ puts the 2 back.

From the physical $\rho$, the paper's number is two multiplications:

* **$\times\,2x$** — it whitens the `jaxgb` template with the fractional-length PSD, so
  every $|\tilde h|^2/S$ is inflated by $4x^2$;
* **$\times\,\Delta t$** — `_optimal_snr_sq` uses `weight = 4.0 * df * dt**2`, right for a
  bare `np.fft.rfft` array ($\tilde h_k = \Delta t X_k$), but the template it is handed is
  `jaxgb.get_tdi` output, which already carries `0.5 * dtm`.

Both factors are in its injected data too: `build_band` adds `signal_rfft` (physical
$\tilde h$) to `noise_rfft` (drawn as $\mathbb{E}[2\,df\,dt^2|X|^2]=S$ from the
fractional-length PSD). Signal, template, likelihood and SNR carry them consistently, so
the paper's WDM-vs-Fourier comparison is unaffected — its injection is simply louder than
the quoted $A$.

In [ ]:
snr_canna = float(problem.snr(p_paper, F_WINDOW))
snr_mf = np.sqrt(2.0) * snr_canna
x0 = 2.0 * np.pi * F0 * L_ARM / C_LIGHT

# the same template, whitened with the paper's fractional-length PSD and its own weight
df = 1.0 / T_OBS
S_paper_win = paper_psd(FREQS)
snr_paper = float(np.sqrt(4.0 * df * DT**2 * np.sum(np.abs(np.asarray(clean)) ** 2 / S_paper_win)))

rows = [
    ("LisaGB.snr(p, f)", snr_canna, "sqrt(<h|h>/2) -- what the repo returns"),
    ("matched filter  sqrt(<h|h>)", snr_mf, "4/T sum |h|^2/S, physical"),
    (f"x 2x  = {2 * x0:.5f}", snr_mf * 2 * x0, "whitened with a fractional-length PSD"),
    (f"x dt  = {DT:.2f}", snr_mf * 2 * x0 * DT, "the paper's 4*df*dt**2 weight"),
    ("same, recomputed directly", snr_paper, "paper PSD + paper weight, canna template"),
]
print(f"{'quantity':<32}{'value':>12}   note")
for name, v, note in rows:
    print(f"{name:<32}{v:>12.4f}   {note}")
print(f"\npaper quotes {SNR_PAPER}; we get {snr_paper:.3f}  "
      f"(relative error {abs(snr_paper / SNR_PAPER - 1):.1%})")

# and it is a converged number, not a response-grid artefact
for npts in (256, 512, 1024):
    q = LisaGB(n_sources=1, t_obs=T_OBS, sampling_step=DT, wdm_freq_bands=256,
               wdm_times=32, response_points=npts, f0_range=(1.3e-3, 1.5e-3))
    print(f"  response_points={npts:>5}  ->  sqrt(2)*snr = "
          f"{np.sqrt(2) * float(q.snr(p_paper, F_WINDOW)):.5f}")

## 5. Checking the units against the response itself

`jaxgb`'s single-link response carries a factor `fonfs` $= 2\pi f L/c \equiv x$
(`jaxgb.py::_construct_slow_part`), and `_compute_tdi_xyz` multiplies by
$-2i\sin x\,e^{-ix}$: `get_tdi` is **fractional frequency**, and its response to a strain
$h$ scales as $x^2$ at low $f$, i.e. $x^4$ in power.

A fractional-length PSD only scales as $x^2$ in power, so pairing the two leaves an $x^2$
tilt. That is a *slope*, which no $\mathcal{O}(1)$ convention factor can fake — which is
what makes it measurable. The test: get the sky- and polarisation-averaged response
$R_c(f) = \langle F_+^2+F_\times^2\rangle$ from `problem.response` itself by Monte Carlo,
form the implied sensitivity $S_h = \left[\sum_c R_c/S_c\right]^{-1}$, and compare with the
standard Robson–Cornish–Liu (2019) LISA curve.

With `noise_psd` as it now stands the ratio is **flat**; dividing it back by $4x^2$ — the
fractional-length form — recovers the $f^{-2}$ tilt.

In [ ]:
M = 256
rng = np.random.default_rng(0)
sky = dict(
    lon=rng.uniform(0, 2 * np.pi, M),
    lat=np.arcsin(rng.uniform(-1, 1, M)),
    psi=rng.uniform(0, 2 * np.pi, M),
    phi0=rng.uniform(0, 2 * np.pi, M),
)

f_grid = np.logspace(np.log10(2e-4), np.log10(3e-2), 16)
R = np.zeros((f_grid.size, 3))
for i, fc in enumerate(f_grid):
    # iota = pi/2 kills the cross polarisation, so the source radiates a pure h+ of
    # amplitude 1; averaging over psi then leaves <F+^2 + Fx^2>/2, hence the 8 (=2*4).
    p = jnp.asarray(
        np.stack([np.full(M, fc), np.zeros(M), np.ones(M), sky["lon"], sky["lat"],
                  sky["psi"], np.full(M, np.pi / 2), sky["phi0"]], axis=-1)
    )
    aet = np.stack([np.asarray(v) for v in problem.response.get_tdi(p, 1.5, "AET")], -1)
    R[i] = 8.0 * np.mean(np.sum(np.abs(aet) ** 2, axis=1), axis=0) / T_OBS**2

S_grid = np.asarray(problem.noise_psd(jnp.asarray(f_grid)))
x = 2.0 * np.pi * f_grid * L_ARM / SPEED_OF_LIGHT
S_h_now = 1.0 / np.sum(R / S_grid, axis=-1)                        # as the code now stands
S_h_len = 1.0 / np.sum(R / (S_grid / (4 * x[:, None] ** 2)), -1)   # the old fractional length

# Robson, Cornish & Liu 2019, the standard sky-averaged LISA sensitivity
fstar = SPEED_OF_LIGHT / (2 * np.pi * L_ARM)
P_oms = (15e-12) ** 2 * (1 + (2e-3 / f_grid) ** 4)
P_acc = (3e-15) ** 2 * (1 + (0.4e-3 / f_grid) ** 2) * (1 + (f_grid / 8e-3) ** 4)
S_rcl = (10 / (3 * L_ARM**2)) * (
    P_oms + 2 * (1 + np.cos(f_grid / fstar) ** 2) * P_acc / (2 * np.pi * f_grid) ** 4
) * (1 + 0.6 * (f_grid / fstar) ** 2)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].loglog(f_grid, S_h_now, "s-", label=r"$S/R$, noise_psd as it stands")
ax[0].loglog(f_grid, S_h_len, "o-", label=r"$S/4x^2$, fractional length")
ax[0].loglog(f_grid, S_rcl, "k--", label="Robson+2019 sensitivity")
ax[0].set(xlabel="f [Hz]", ylabel=r"$S_h(f)$ [strain$^2$/Hz]", title="implied LISA sensitivity")
ax[0].legend(fontsize=8)
ax[1].loglog(f_grid, S_h_now / S_rcl, "s-", label="as it stands")
ax[1].loglog(f_grid, S_h_len / S_rcl, "o-", label=r"$/\,4x^2$")
ax[1].axhline(0.5, color="k", ls=":", lw=1)
ax[1].set(xlabel="f [Hz]", ylabel="ratio to Robson+2019", title="a flat ratio means consistent units")
ax[1].legend(fontsize=8)
fig.tight_layout()

ratio_now, ratio_len = S_h_now / S_rcl, S_h_len / S_rcl
below = f_grid < 4e-3
print(f"slope of the ratio below 3 mHz : {np.polyfit(np.log(f_grid[:8]), np.log(ratio_now[:8]), 1)[0]:+.3f}"
      f"   (fractional length would give {np.polyfit(np.log(f_grid[:8]), np.log(ratio_len[:8]), 1)[0]:+.2f})")
print(f"ratio below 4 mHz              : {ratio_now[below].min():.4f} .. {ratio_now[below].max():.4f}")
print("   -> flat at 1/2, the gain of A+E over one sky-averaged channel. The drift above")
print("      ~8 mHz is the Robson+2019 curve's own (1 + 0.6 x^2) approximation, not the PSD.")
print(f"\n{'f [Hz]':>10}{'as it stands':>15}{'fractional length':>20}")
for fc, a, b in zip(f_grid, ratio_now, ratio_len):
    print(f"{fc:>10.3e}{a:>15.4f}{b:>20.3f}")

## 6. Reproducing Fig. 6

Set-up copied from the paper:

* sample $(f_0,\dot f, g_c, g_s)$ with $g_c = A\cos\phi_0$, $g_s = A\sin\phi_0$; sky and
  orientation held at the injected values;
* Gaussian priors $\sigma_{f_0} = 0.32/T$, $\sigma_{\dot f} = 10/T^2$, recentred on the
  truth; isotropic Gaussian on $(g_c,g_s)$ — a Rayleigh prior on $A$, uniform on $\phi_0$;
* the sampler is preconditioned by the inverse Fisher, as the paper's NUTS mass matrix is.

What sets the posterior widths is the **matched-filter SNR**, so the amplitude is solved
for $\rho = 20.65$ rather than hardcoded — that keeps this cell correct under either PSD
convention. It lands near $1.3\times10^{-22}$, a perfectly ordinary GB amplitude at
1.4 mHz; the paper's quoted $5.35\times10^{-24}$ would be $\rho=0.85$. The corner's
amplitude axis is rescaled by the same ratio, so all four panels overlay Fig. 6.

The template is `LisaGB.clean_signal`, reached through `chirp_mass_from_fdot` since
column 1 is a chirp mass; the cell checks the resulting likelihood against
`LisaGB.log_likelihood` exactly. That inverse is undefined for $\dot f < 0$, which the
broad $\dot f$ prior formally allows — harmless here, because the posterior sits $21\sigma$
away from zero and a NaN log-posterior is rejected by the Metropolis test anyway.

In [ ]:
SNR_TARGET = 20.645
# snr is linear in amplitude, so one evaluation fixes the scale
A_INJ = AMP * SNR_TARGET / (np.sqrt(2) * float(problem.snr(p_paper, F_WINDOW)))
AMP_RESCALE = AMP / A_INJ  # maps our amplitude back onto the paper's axis
p_true = source(amp=A_INJ)
print(f"paper's quoted A  : {AMP:.4e}  ->  matched-filter SNR "
      f"{np.sqrt(2) * float(problem.snr(p_paper, F_WINDOW)):.4f}")
print(f"injected A        : {A_INJ:.4e}  ->  matched-filter SNR "
      f"{np.sqrt(2) * float(problem.snr(p_true, F_WINDOW)):.4f}")

WHITEN = jax.lax.rsqrt(problem.noise_psd(jnp.asarray(FREQS)) * T_OBS / 2.0)


@jax.jit
def template(f0, fdot):
    """Whitened unit-amplitude, zero-phase A/E/T response -- same path as clean_signal."""
    mc = chirp_mass_from_fdot(fdot, f0)
    p = jnp.stack([f0, mc, jnp.ones(()), jnp.asarray(LON), jnp.asarray(LAT),
                   jnp.asarray(PSI), jnp.pi / 2 - jnp.asarray(IOTA), jnp.zeros(())])[None]
    return problem.clean_signal(p, F_WINDOW) * WHITEN


# jaxgb's phi0 enters as a pure phase, so the waveform is exactly linear in (g_c, g_s):
# h(A, phi0) = (g_c - i g_s) * h(1, 0). Check it, then use it.
h1 = template(jnp.asarray(F0), jnp.asarray(FDOT))
h2 = h1 * jnp.exp(-1j * PHI0)
h3 = problem.clean_signal(source(amp=1.0, phi0=PHI0), F_WINDOW) * WHITEN
print(f"|h(phi0) - e^-i.phi0 h(0)| / |h| : {float(jnp.abs(h3 - h2).max() / jnp.abs(h1).max()):.2e}")

data = problem.sample_observation(jr.key(20269), p_true, F_WINDOW)
DATA_W = data * WHITEN


@jax.jit
def log_likelihood(theta):
    f0, fdot, gc, gs = theta
    residual = DATA_W - (gc - 1j * gs) * template(f0, fdot)
    return -jnp.sum(jnp.abs(residual) ** 2)


theta_true = jnp.array([F0, FDOT, A_INJ * np.cos(PHI0), A_INJ * np.sin(PHI0)])
print(f"logL(truth) here                : {float(log_likelihood(theta_true)):.6f}")
print(f"LisaGB.log_likelihood(truth)    : {float(problem.log_likelihood(p_true, data, F_WINDOW)):.6f}")

In [ ]:
SIG_F0, SIG_FDOT = 0.32 / T_OBS, 10.0 / T_OBS**2
SIG_G = 10.0 * A_INJ  # far wider than the posterior, as in the paper
print(f"prior sigmas: f0 {SIG_F0:.3e} Hz, fdot {SIG_FDOT:.3e} Hz/s")


@jax.jit
def log_posterior(theta):
    f0, fdot, gc, gs = theta
    log_prior = (
        -0.5 * ((f0 - F0) / SIG_F0) ** 2
        - 0.5 * ((fdot - FDOT) / SIG_FDOT) ** 2
        - 0.5 * (gc**2 + gs**2) / SIG_G**2
    )
    return log_likelihood(theta) + log_prior


# Fisher at the truth: <d_i h | d_j h> = 2 Re sum (d_i h_w)(d_j h_w)*, plus the prior
# precision. Central differences keep this off jaxgb's rint()-quantised kmin.
STEPS = jnp.array([1e-3 / T_OBS, 1e-3 / T_OBS**2, 1e-3 * A_INJ, 1e-3 * A_INJ])


def waveform(theta):
    f0, fdot, gc, gs = theta
    return (gc - 1j * gs) * template(f0, fdot)


derivatives = jnp.stack([
    (waveform(theta_true + jnp.zeros(4).at[i].set(STEPS[i]))
     - waveform(theta_true - jnp.zeros(4).at[i].set(STEPS[i]))) / (2 * STEPS[i])
    for i in range(4)
])
fisher = 2.0 * jnp.real(jnp.einsum("ifc,jfc->ij", derivatives, jnp.conj(derivatives)))
fisher += jnp.diag(jnp.array([SIG_F0**-2, SIG_FDOT**-2, SIG_G**-2, SIG_G**-2]))
covariance = jnp.linalg.inv(fisher)

print(f"Fisher sigma(f0)   = {float(covariance[0, 0]) ** 0.5:.3e} Hz   "
      f"({float(covariance[0, 0]) ** 0.5 / SIG_F0:.2f} x prior)")
print(f"Fisher sigma(fdot) = {float(covariance[1, 1]) ** 0.5:.3e} Hz/s "
      f"({float(covariance[1, 1]) ** 0.5 / SIG_FDOT:.3f} x prior -- 'tens of times narrower')")

In [ ]:
N_STEP, N_CHAIN = 40_000, 4
proposal = jnp.linalg.cholesky(covariance) * 2.38 / np.sqrt(4)


def run_chain(key, theta0):
    """Fisher-preconditioned random-walk Metropolis."""

    def step(carry, key):
        theta, logp = carry
        key_move, key_accept = jr.split(key)
        proposed = theta + proposal @ jr.normal(key_move, (4,))
        logp_proposed = log_posterior(proposed)
        accept = jnp.log(jr.uniform(key_accept)) < logp_proposed - logp
        theta, logp = jax.lax.cond(
            accept, lambda: (proposed, logp_proposed), lambda: (theta, logp)
        )
        return (theta, logp), (theta, accept)

    _, (chain, accepted) = jax.lax.scan(
        step, (theta0, log_posterior(theta0)), jr.split(key, N_STEP)
    )
    return chain, accepted


starts = theta_true + jax.vmap(lambda k: proposal @ jr.normal(k, (4,)))(
    jr.split(jr.key(1), N_CHAIN)
)
chains, accepted = jax.block_until_ready(
    jax.vmap(run_chain)(jr.split(jr.key(2), N_CHAIN), starts)
)
print(f"{N_CHAIN} chains x {N_STEP} steps, acceptance {float(accepted.mean()):.2f}")

kept = np.asarray(chains[:, N_STEP // 2 :, :])  # drop the first half as warm-up
between = kept.mean(1).var(0, ddof=1) * kept.shape[1]
within = kept.var(1, ddof=1).mean(0)
rhat = np.sqrt(((kept.shape[1] - 1) / kept.shape[1] * within + between / kept.shape[1]) / within)
print(f"R-hat: {np.array2string(rhat, precision=4)}   (paper requires < 1.01)")

In [ ]:
f0_s, fdot_s, gc_s, gs_s = kept.reshape(-1, 4).T
amp_s = np.hypot(gc_s, gs_s)
phi_s = np.mod(np.arctan2(gs_s, gc_s), 2 * np.pi)

# Fig. 6's axes: f0 - f0_inj in nHz, fdot in 1e-15 Hz/s, log10 A on the paper's
# normalisation (AMP_RESCALE puts our louder injection back on its axis), phi0
samples = np.stack([
    (f0_s - F0) * 1e9,
    fdot_s * 1e15,
    np.log10(amp_s * AMP_RESCALE),
    phi_s,
], axis=-1)
truths = [0.0, FDOT * 1e15, np.log10(AMP), PHI0]
labels = [r"$f_0-f_0^{\rm inj}$ [nHz]", r"$\dot f$ [$10^{-15}$ Hz/s]",
          r"$\log_{10} A$", r"$\phi_0$ [rad]"]

print(f"{'parameter':<26}{'truth':>12}{'mean':>12}{'sigma':>11}{'z':>7}")
for i, lab in enumerate(["f0 - f0_inj [nHz]", "fdot [1e-15 Hz/s]", "log10 A", "phi0 [rad]"]):
    col = samples[:, i]
    z = (col.mean() - truths[i]) / col.std()
    print(f"{lab:<26}{truths[i]:>12.4f}{col.mean():>12.4f}{col.std():>11.4f}{z:>+7.2f}")

fig = corner.corner(
    samples, labels=labels, truths=truths, truth_color="k",
    color="tab:blue", bins=40, smooth=1.0, plot_datapoints=False,
    levels=(0.393, 0.865, 0.989), hist_kwargs={"density": True, "lw": 1.4},
    label_kwargs={"fontsize": 11},
)

# the priors, dotted, as in Fig. 6 -- flat across every posterior except f0's
axes = np.asarray(fig.axes).reshape(4, 4)
prior = [
    lambda v: np.exp(-0.5 * (v / (SIG_F0 * 1e9)) ** 2),
    lambda v: np.exp(-0.5 * ((v - FDOT * 1e15) / (SIG_FDOT * 1e15)) ** 2),
    None,
    None,
]
for i, fn in enumerate(prior):
    ax = axes[i, i]
    lo, hi = ax.get_xlim()
    grid = np.linspace(lo, hi, 200)
    curve = fn(grid) if fn is not None else np.ones_like(grid)
    ax.plot(grid, curve / curve.max() * ax.get_ylim()[1] * 0.95, "k:", lw=1.2)
fig.suptitle(f"canna.lisa, matched-filter SNR = {np.sqrt(2) * float(problem.snr(p_true, F_WINDOW)):.1f}"
             f"  (arXiv:2606.20269 Fig. 6 quotes 20.7)", y=1.01)

## Conclusions

| question | answer |
|---|---|
| Same instrument model as the paper? | **Yes** — the same 3 fm/s² / 15 pm TDI-1.5 A/E/T noise, and the same `jaxgb` TDI-1.5 AET waveform on equal-arm orbits |
| Is `noise_psd` in the response's units? | **Yes, now** — fractional frequency, equal to the textbook TDI-1.5 A/E/T PSD to machine precision. It used to be fractional length, out by $4(2\pi fL/c)^2$ |
| Noise and datastream consistent *inside* `canna.lisa`? | **Yes** — $\mathbb{E}\lvert n\rvert^2=ST/2$ in `sample_observation`, `log_likelihood` and `preprocess`; $\chi^2/\mathrm{dof}=1$ |
| Is `LisaGB.snr` the matched-filter SNR? | **No** — it is $\sqrt{\langle h\lvert h\rangle/2}$; multiply by $\sqrt2$ |
| Where does the paper's 20.7 come from? | $\rho=0.85$ physically, $\times\,2x$ (fractional-length whitening) $\times\,\Delta t$ (`4*df*dt**2` on a template that is already a physical $\tilde h$) |

The units fix is the one that changes training: it is **frequency dependent**, so it
re-weights the band rather than rescaling it. Across `f0_range=(1e-4, 12e-3)` the SNR of a
fixed source moves by $1/(2x)$ — $\times95$ at 0.1 mHz, $\times9.5$ at 1 mHz, $\times3.2$
at 3 mHz, $\times1$ near 6.6 mHz, $\times0.8$ at 12 mHz. Low-frequency sources are now far
louder relative to high-frequency ones, so the `a_range` prior and the SNR distribution the
network trains on are worth revisiting.